# TargetAgeGroup: attribute, co-occurrence, and sparse-data risk

Audit the claim that TargetAgeGroup is a product attribute rather than a level of the Segment/Sub-Segment taxonomy. Verify label counts, inspect Pregnancy Vitamins, measure many-to-many associations, and test empirical co-occurrence rules on held-out product groups.

The supplied CSVs are read only. Valid values for this audit are Adult, Children, and Infant, as specified in the claim. Observed co-occurrence cannot establish real-world suitability, clinical recommendations, or ontological ownership. No labels are automatically corrected.


## Findings from this run

- Valid counts match: Adult **72,995**, Children **3,718**, Infant **759**. There are also **7,077 missing** and **3 invalid nonblank** training values.
- Pregnancy Vitamins has **359 rows: 302 Adult (84.12%), 53 Infant, and 4 Children**. It skews Adult but does not meet a 90% Adult rule in the full sample. Non-Adult cases require review, not automatic relabeling.
- The 759 Infant rows represent **689 candidate product groups** under the matching heuristic.
- **19 Sub-Segments** have no observed Infant among valid-age records. These are sample zeros, not forbidden combinations.
- Strong Sub-Segment rules cover **78.66%** of held-out valid-age rows, with **96.37%** selective accuracy. The majority-age baseline on those same rows scores **96.37%**. Inspect per-age results before treating this as useful performance.
- **0 held-out Infant rows** occur in Sub-Segments that had no Infant in the fitting sample.

The high selective accuracy provides no gain over always predicting Adult on the covered rows. Zero held-out counterexamples in this split do not establish impossibility.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib, re
ROOT=Path.cwd()
if not (ROOT/'data'/'provided').exists(): ROOT=ROOT.parent
DATA=ROOT/'data'/'provided'
train=pd.read_csv(DATA/'Selfcare_Training_data (1).csv',low_memory=False)
target=pd.read_csv(DATA/'Selfcare_Target_data (1).csv',low_memory=False)
AGES=['Adult','Children','Infant']
def present(s): return s.notna() & s.astype('string').str.strip().ne('').fillna(False)
def normalized(s): return s.astype('string').str.strip()
train['_age']=normalized(train['TargetAgeGroup'])
valid=train.loc[train['_age'].isin(AGES)].copy()
for col in ['Segment','Sub-Segment']:
    valid[col]=normalized(valid[col])
print('Training rows:',len(train),'Target rows:',len(target))


Training rows: 84552 Target rows: 28082


## 1. Verify valid labels, missingness, and suspicious values

In [2]:
for name,df in [('training',train),('target',target)]:
    age=normalized(df['TargetAgeGroup'])
    print('\n'+name+' raw values:'); print(df['TargetAgeGroup'].value_counts(dropna=False).to_string())
    print('Valid:',int(age.isin(AGES).sum()),'missing/blank:',int((~present(df['TargetAgeGroup'])).sum()),
          'nonblank invalid:',int((present(df['TargetAgeGroup']) & ~age.isin(AGES)).sum()))
print('\nValid training class proportions:')
print(valid['_age'].value_counts().reindex(AGES).to_frame('rows').assign(pct=lambda x:100*x.rows/x.rows.sum()).to_string())
print('\nInvalid training rows, for review:')
print(train.loc[present(train['TargetAgeGroup']) & ~train['_age'].isin(AGES),
 ['Brand','Segment','Sub-Segment','TargetAgeGroup','ProductName']].to_string(max_colwidth=120))



training raw values:
TargetAgeGroup
Adult                                                                                                                                      72995
NaN                                                                                                                                         7077
Children                                                                                                                                    3718
Infant                                                                                                                                       759
 Paleo friendly                                                                                                                                1
 starch or artificial flavors; 3 tablets daily"                                                                                                1
 starch or artificial flavors; 3 tablets daily NeoCell Super Collagen + Vitamin C and Biotin 

## 2. Does the observed structure resemble a hierarchy?

Cross-tabulate each taxonomy field with age. Multiple ages per Segment contradict a deterministic Segment-to-age assignment in these annotations. Multiple Segments per age show that age spans taxonomy categories. This supports modeling age separately but cannot, by itself, prove a domain ontology.


In [3]:
tables={}
for col in ['Segment','Sub-Segment']:
    subset=valid.loc[present(valid[col])]
    tab=pd.crosstab(subset[col],subset['_age']).reindex(columns=AGES,fill_value=0)
    tables[col]=tab
    print('\n'+col+' counts by age:'); print(tab.to_string())
    print('Groups with >1 observed age:',int(tab.gt(0).sum(axis=1).gt(1).sum()),'of',len(tab))
    print('Number of taxonomy groups per age:'); print(tab.gt(0).sum().to_string())
    print('Within-group percentages:'); print(tab.div(tab.sum(axis=1),axis=0).mul(100).round(2).to_string())



Segment counts by age:
_age                              Adult  Children  Infant
Segment                                                  
Allergy                            1969       321      27
CCFS                               3698       462     106
Digestive Health                   6280       387     105
External Analgesics                5787       167      72
Internal Analgesics                4469       249      86
Lifestyle CHC                      4131       356      63
Other Self Care                    1124        10       1
Vitamins, Minerals & Supplements  44971      1747     288
Groups with >1 observed age: 8 of 8
Number of taxonomy groups per age:
_age
Adult       8
Children    8
Infant      8
Within-group percentages:
_age                              Adult  Children  Infant
Segment                                                  
Allergy                           84.98     13.85    1.17
CCFS                              86.69     10.83    2.48
Digestive Health    

## 3. Pregnancy Vitamins: inspect the actual exception labels

?Should skew Adult? is an expectation supplied in the claim, not a medical conclusion established here. Count all 359 rows, inspect non-Adult product names, and preserve source-row indices for review. Do not relabel products using the Sub-Segment alone.


In [4]:
preg=train.loc[normalized(train['Sub-Segment']).eq('Pregnancy Vitamins')].copy()
print('Pregnancy Vitamins rows:',len(preg))
print(preg['TargetAgeGroup'].value_counts(dropna=False).to_string())
print('Adult percentage of all Pregnancy Vitamins rows:',100*preg['_age'].eq('Adult').mean())
print('\nEvery non-Adult annotation (source-row index retained):')
print(preg.loc[~preg['_age'].eq('Adult'),['Retailer','Brand','ProductName','ProductCategory','TargetAgeGroup']].to_string(max_colwidth=130))


Pregnancy Vitamins rows: 359
TargetAgeGroup
Adult       302
Infant       53
Children      4
Adult percentage of all Pregnancy Vitamins rows: 84.12256267409471

Every non-Adult annotation (source-row index retained):
      Retailer               Brand                                                                                                                        ProductName                                                                                           ProductCategory TargetAgeGroup
891     Amazon    UnItemised brand  Klaire Labs Prenatal & Nursing Formula - Multivitamin & Multimineral with Chelated Iron & Active Folate for Pregnancy, Breastf...                      Health & Household > Vitamins, Minerals & Supplements > Vitamins > Prenatal Vitamins       Children
7872    Amazon         Nature Made  Nature Made Prenatal Multi, 90 Tablets, Folic Acid + 17 Prenatal Vitamins & Minerals to Support Baby Development and Mom, Vita...                      Health & Household > Vi

## 4. Product groups: quantify repeat observations and reduce holdout leakage

Connect rows sharing nonblank normalized UPC, URL, or exact product name, including transitive matches. These are candidate product identities, not verified identities. Bad shared identifiers can over-group; missing or changed identifiers can under-group. Row counts can overstate independent evidence.


In [5]:
parent=list(range(len(train)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]]; x=parent[x]
    return x
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
for col in ['Upc','ProductUrl','ProductName']:
    seen={}
    for idx,v in train[col].items():
        if pd.isna(v): continue
        key=re.sub(r'\s+',' ',str(v)).strip().casefold()
        if not key: continue
        if key in seen: union(idx,seen[key])
        else: seen[key]=idx
valid['_group']=[find(i) for i in valid.index]
preg['_group']=[find(i) for i in preg.index]
print('Rows versus candidate product groups by age:')
print(valid.groupby('_age').agg(rows=('_age','size'),groups=('_group','nunique')).reindex(AGES).to_string())
print('Pregnancy Vitamins rows versus groups:')
print(preg.groupby('_age').agg(rows=('_age','size'),groups=('_group','nunique')).to_string())
conflict=valid.groupby('_group')['_age'].nunique()
print('Groups with conflicting age annotations:',int(conflict.gt(1).sum()))
print('Pregnancy groups with >1 age:',int(preg.groupby('_group')['_age'].nunique().gt(1).sum()))


Rows versus candidate product groups by age:
           rows  groups
_age                   
Adult     72995   67786
Children   3718    3438
Infant      759     689
Pregnancy Vitamins rows versus groups:
          rows  groups
_age                  
Adult      302     282
Children     4       4
Infant      53      53
Groups with conflicting age annotations: 123
Pregnancy groups with >1 age: 0


## 5. ?Never Infant? in this sample is not an impossibility rule

For each Sub-Segment, count valid-age rows, Infant rows, and distinct candidate product groups. For zero-Infant groups, report the illustrative one-sided 95% binomial upper bound `1 - 0.05**(1/n)` using the number of candidate product groups. The calculation assumes independent, representative observations, which this dataset may not satisfy. It quantifies why a small zero count is weak evidence, not a population guarantee.


In [6]:
sub=valid.loc[present(valid['Sub-Segment'])]
stats=sub.groupby('Sub-Segment').agg(rows=('_age','size'),groups=('_group','nunique'),infant_rows=('_age',lambda x:x.eq('Infant').sum()))
zero=stats.loc[stats['infant_rows'].eq(0)].copy()
zero['illustrative_upper_95_pct']=100*(1-0.05**(1/zero['groups']))
print('Sub-Segments with zero observed Infant:',len(zero),'of',len(stats))
print(zero.sort_values('groups').to_string())
print('\nInfant concentration across Sub-Segments:')
infant_counts=sub.loc[sub['_age'].eq('Infant'),'Sub-Segment'].value_counts()
print(infant_counts.to_string())
print('Share of Infant rows in top 3 observed Sub-Segments (%):',100*infant_counts.head(3).sum()/valid['_age'].eq('Infant').sum())


Sub-Segments with zero observed Infant: 19 of 51
                           rows  groups  infant_rows  illustrative_upper_95_pct
Sub-Segment                                                                    
Creams & Gels (Rubs)          1       1            0                  95.000000
Daily oral contracception     2       2            0                  77.639320
Other Lifestyle               3       3            0                  63.159685
Dermatologicals               4       4            0                  52.712920
Cold / Flu                    4       4            0                  52.712920
Stimulants                   10      10            0                  25.886555
Remaining Internal Pain      11      11            0                  23.840419
Devices/Refills              14      14            0                  19.263618
Topical Itch                 32      31            0                   9.211406
Lactose Intolerance          42      37            0                   

## 6. Test soft co-occurrence on held-out product groups

Hash candidate groups into an approximately 80/20 fit/validation split. Learn Sub-Segment majority-age rules only on fit rows, requiring at least 20 rows, 10 candidate groups, and 90% agreement. Thresholds are fixed before evaluation. These are diagnostic age rules, not a hierarchy or recommended product-use guidance.

Report coverage, accuracy, and disagreement separately for each true age. Adult dominance can make overall accuracy misleading. Separately count held-out Infant rows in Sub-Segments with zero Infant in fitting data: these directly challenge treating an observed zero as impossible. Target coverage is not assessed as a model deployment claim because its Sub-Segment labels are also absent.


In [7]:
holdout=valid['_group'].map(lambda g:int(hashlib.sha256(str(g).encode()).hexdigest()[:8],16)%5==0)
fit=valid.loc[~holdout].copy(); val=valid.loc[holdout].copy()
assert set(fit['_group']).isdisjoint(val['_group'])
fit_sub=fit.loc[present(fit['Sub-Segment'])]
ct=pd.crosstab(fit_sub['Sub-Segment'],fit_sub['_age']).reindex(columns=AGES,fill_value=0)
rule_table=pd.DataFrame({'support':ct.sum(axis=1),'groups':fit_sub.groupby('Sub-Segment')['_group'].nunique(),
                         'prediction':ct.idxmax(axis=1),'purity':ct.max(axis=1)/ct.sum(axis=1)})
rules=rule_table.loc[rule_table.support.ge(20)&rule_table.groups.ge(10)&rule_table.purity.ge(.9)]
pred=val['Sub-Segment'].map(rules['prediction'])
covered=pred.notna(); correct=pred.eq(val['_age'])
majority=fit['_age'].value_counts().index[0]
print('Fit/validation rows:',len(fit),len(val))
print('Qualifying rules:',len(rules)); print(rules.to_string())
metrics={'coverage_pct':100*covered.mean(),'accuracy_on_covered_pct':100*correct[covered].mean(),
 'majority_baseline_same_covered_pct':100*val.loc[covered,'_age'].eq(majority).mean()}
print('\nOverall metrics:'); print(pd.Series(metrics).to_string())
per_age=[]
for age in AGES:
    mask=val['_age'].eq(age); selected=mask&covered
    per_age.append({'age':age,'validation_rows':int(mask.sum()),'covered':int(selected.sum()),
      'coverage_pct':100*covered[mask].mean(),'accuracy_on_covered_pct':100*correct[selected].mean(),
      'review_candidates':int((selected&~correct).sum())})
print('\nPer-age outcomes:'); print(pd.DataFrame(per_age).to_string(index=False))
zero_in_fit=ct.index[ct['Infant'].eq(0)]
counterexamples=val.loc[val['_age'].eq('Infant') & val['Sub-Segment'].isin(zero_in_fit)]
print('\nHeld-out Infant rows in previously observed Sub-Segments with zero fit Infant:',len(counterexamples))
print(counterexamples[['Sub-Segment','ProductName','TargetAgeGroup']].to_string(max_colwidth=100))
print('\nExamples contradicting strong rules (review candidates, not established label errors):')
print(val.loc[covered & ~correct,['Sub-Segment','ProductName','TargetAgeGroup']].assign(rule_prediction=pred[covered & ~correct]).head(25).to_string(max_colwidth=110))


Fit/validation rows: 62022 15450
Qualifying rules: 28
                                 support  groups prediction    purity
Sub-Segment                                                          
Acid Relief                          925     766      Adult  0.988108
All Other Digestive Health          1702    1655      Adult  0.903643
Allergy INS                          485     450      Adult  0.936082
Anti-Diarrheal                       186     164      Adult  0.956989
Cold & Heat Wraps                   3042    2865      Adult  0.962853
Constipation Remedies               1407    1305      Adult  0.949538
Creams/Gels & Medicated Patches      813     800      Adult  0.982780
Digestive enzymes                    306     297      Adult  0.990196
Eye Care                             822     708      Adult  0.998783
Hemorrhoid Remedies                  176     172      Adult  1.000000
Herbal & Natural (H&N)              8404    8120      Adult  0.970847
Lactose Intolerance                 

## Interpretation and safe use of these findings

Age is modeled as a separate attribute in this audit. Cross-tab associations do not create parent-child ownership between age and Segment. Sparse or absent combinations should remain possible unless a trusted domain rule says otherwise.

Pregnancy Vitamins exceptions need annotation review with product evidence and labeling definitions. The audit does not establish whether any individual age label is correct or medically appropriate. A broad majority rule should flag a discrepancy for review, not replace the supplied age.

The 759 Infant rows are not necessarily 759 independent products. Product-group counts, concentration, and per-age holdout results are more informative than total accuracy. Even a large, perfectly consistent observed group cannot prove that an unobserved combination is impossible in the world.
